In [1]:
from tools import *
from differential import *
from pprint import pprint

MAKE SURE EVERYTHING IS IN CGS


In [3]:
sim_path = "/Users/reza/Career/DMLab/SURROGATE/sun-sim/src/sfno/mhd/vr_rho_pinn/cr2239-all-components/hmi_masp_mas_std_0201"
system = "cgs"
mas, coefficients, theta, phi = get_sim(sim_path, system=system)
r = mas['r']
print(type(r), "dtype of r")

br shape after interpolation and transpose:  (140, 111, 128)
bt shape after interpolation and transpose:  (140, 111, 128)
bp shape after interpolation and transpose:  (140, 111, 128)
jr shape after interpolation and transpose:  (140, 111, 128)
jt shape after interpolation and transpose:  (140, 111, 128)
jp shape after interpolation and transpose:  (140, 111, 128)
<class 'torch.Tensor'> dtype of r


In [4]:
pprint(coefficients)
print(system, "system\n", "Nr:", r.shape, "Nt:", theta.shape, "Np:", phi.shape)

{'G': 6.6743e-08,
 'M': 1.9884999999999998e+33,
 'c': 29979245800.0,
 'nu': 1.675171314428855e+16,
 'omega_rot': 2.84e-06}
cgs system
 Nr: torch.Size([140]) Nt: torch.Size([111]) Np: torch.Size([128])


In [5]:
result = ampere_residual(
    mas["br"],
    mas["bt"],
    mas["bp"],
    mas["jr"],
    mas["jt"],
    mas["jp"],
    r,
    theta,
    dr=r[1] - r[0],
    dtheta=theta[1] - theta[0],
    dphi=phi[1] - phi[0],
    c=coefficients["c"]
)

In [6]:
result.keys()

dict_keys(['lhs', 'rhs', 'residual'])

In [7]:
mask = build_analysis_mask(result['lhs'][0], n_exclude_theta=2)

In [8]:
metrics = detailed_residual_metrics(
    result["lhs"],
    result["rhs"],
    result["residual"],
    mask=None
)

print(metrics)

{'RMS_LHS': 1.857920021766022e-08, 'RMS_RHS': 2.1396016907183878e-16, 'RMS_RES': 1.8579200218720902e-08, 'RMS_RES/LHS': 1.0000000000570897, 'RMS_RES/RHS': 86834854.82049093, 'rel_p50': 0.013508999974137587, 'rel_p90': 0.12261090235603528, 'rel_p95': 0.20318476937121918, 'rel_p99': 0.9999999997293249, 'rel_max': 0.9999999999999053}


In [9]:
radial_profile = radial_rms_profile(result["residual"], mask)
# plot_radial_profile(radial_profile)

make_residual_gif(
    "residual.gif",
    *result["residual"],
    mask
)

100%|██████████| 140/140 [00:08<00:00, 17.17it/s]


Saved: residual.gif


In [10]:
make_lhs_rhs_residual_gif(
    "lhs_rhs.gif", result["lhs"], result["rhs"], result["residual"], mask
)

100%|██████████| 140/140 [00:25<00:00,  5.39it/s]


Saved: lhs_rhs.gif


In [11]:
df = full_numerical_analysis(
    result["lhs"],
    result["rhs"],
    result["residual"],
    r=r,
    theta=theta,
    phi=phi,
    mask=mask,
    volume_weighted=False
)

print(df)


                         Value
N_points          1.917440e+06
L2_LHS            2.074922e-16
L2_RHS            2.178686e-16
L2_Residual       2.367000e-17
L2_Residual/LHS   1.140766e-01
Mean_|LHS|        4.302500e-17
Mean_|RHS|        4.478706e-17
Mean_|Res|        4.178199e-18
Max_|Res|         1.784072e-15
Median_rel_error  1.230553e-02
P90_rel_error     1.082024e-01
P95_rel_error     1.755682e-01
P99_rel_error     3.324555e-01
Max_rel_error     9.995356e-01


In [12]:
term_budget_table(result, mask=mask)


Term budgets (masked interior)
          term |        rms |    mean|x| |     p95|x| |     p99|x| |     max|x|
-------------------------------------------------------------------------------
          lhs_r |  3.907e-17 |  1.183e-17 |  4.405e-17 |  1.515e-16 |  1.772e-15
          lhs_t |  1.163e-16 |  2.153e-17 |  7.763e-17 |  3.943e-16 |  7.245e-15
          lhs_p |  1.673e-16 |  2.905e-17 |  9.549e-17 |  5.349e-16 |  1.085e-14
        lhs_mag |  2.075e-16 |  4.303e-17 |  1.471e-16 |  7.455e-16 |  1.100e-14
          rhs_r |  4.232e-17 |  1.233e-17 |  4.543e-17 |  1.593e-16 |  1.982e-15
          rhs_t |  1.180e-16 |  2.223e-17 |  8.259e-17 |  4.087e-16 |  7.930e-15
          rhs_p |  1.782e-16 |  3.021e-17 |  9.855e-17 |  5.465e-16 |  1.261e-14
        rhs_mag |  2.179e-16 |  4.479e-17 |  1.543e-16 |  7.600e-16 |  1.278e-14
     residual_r |  5.812e-18 |  1.095e-18 |  4.440e-18 |  1.988e-17 |  3.488e-16
     residual_t |  9.315e-18 |  9.280e-19 |  2.809e-19 |  2.565e-17 |  6.929e-16

[('lhs_r',
  3.906784415785214e-17,
  1.1827996367748777e-17,
  4.40482790532082e-17,
  1.5150693354517659e-16,
  1.7724900104583277e-15),
 ('lhs_t',
  1.1633714485930358e-16,
  2.1527638745772847e-17,
  7.76291255261475e-17,
  3.943478703611005e-16,
  7.244856818798023e-15),
 ('lhs_p',
  1.6730924805793256e-16,
  2.905069580599679e-17,
  9.549004238640199e-17,
  5.349465398942591e-16,
  1.085383563835314e-14),
 ('lhs_mag',
  2.074921979421579e-16,
  4.3025000918018735e-17,
  1.470734766591039e-16,
  7.455431382062193e-16,
  1.1001073958854537e-14),
 ('rhs_r',
  4.232027218204221e-17,
  1.2331316879438198e-17,
  4.542876334782164e-17,
  1.593095131457583e-16,
  1.9820850651440298e-15),
 ('rhs_t',
  1.1803758132219212e-16,
  2.223048938827199e-17,
  8.258531137780514e-17,
  4.087013540791043e-16,
  7.930001878284253e-15),
 ('rhs_p',
  1.7816517475036963e-16,
  3.0208164765260127e-17,
  9.855233056085418e-17,
  5.464636710265823e-16,
  1.2614441044437713e-14),
 ('rhs_mag',
  2.1786855104